# Axis-aware ZarrManager example

This notebook shows how to write and read OME-Zarr data using axes other than legacy `tczyx`.

In [ ]:
from pathlib import Path
import dask.array as da
import numpy as np
import pymif.microscope_manager as mm

out_dir = Path('examples_output')
out_dir.mkdir(exist_ok=True)

## Write and read a 2D YX intensity image

In [ ]:
image = np.arange(256 * 384, dtype=np.uint16).reshape(256, 384)
levels = [da.from_array(image, chunks=(128, 128)), da.from_array(image[::2, ::2], chunks=(64, 64))]
metadata = {
    'axes': 'yx',
    'size': [level.shape for level in levels],
    'chunksize': [level.chunksize for level in levels],
    'scales': [(0.5, 0.5), (1.0, 1.0)],
    'units': ('micrometer', 'micrometer'),
    'dtype': 'uint16',
    'data_type': 'intensity',
}
path = out_dir / 'widefield_yx.zarr'
mm.ArrayManager(levels, metadata).to_zarr(path, ngff_version='0.5', zarr_format=3, overwrite=True)
z = mm.ZarrManager(path)
z.metadata

## Create labels and subset without T/C axes

In [ ]:
labels = np.zeros_like(image, dtype=np.uint16)
labels[50:120, 80:170] = 1
labels[130:220, 200:320] = 2
label_levels = [da.from_array(labels, chunks=(128, 128)), da.from_array(labels[::2, ::2], chunks=(64, 64))]
label_metadata = {**metadata, 'size': [level.shape for level in label_levels], 'chunksize': [level.chunksize for level in label_levels], 'data_type': 'label'}
z = mm.ZarrManager(path, mode='a')
z.create_empty_group('segmentation', label_metadata, data_type='label')
z.subset_dataset(Y=slice(32, 160), X=slice(64, 192), include_groups=False, include_labels=False)
z.data[0].shape